# Notebook 02: Surrogate Emulator Training

**Gap 5**: The ray-tracing code requires 3.5×10⁸ photon packages per configuration.
A trained neural emulator reproduces spectra ~1000× faster, enabling real-time fitting.


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import torch
import matplotlib.pyplot as plt
from src.emulator.model import SpectralEmulator, ParameterNormalizer, load_emulator
from src.utils.physics import PARAM_BOUNDS, ENERGY_BINS
from src.utils.io import load_dataset_hdf5


In [ ]:
# Load dataset generated in Notebook 01
X, y = load_dataset_hdf5('../data/simulated/demo_sweep.h5')
print(f'Spectra: {X.shape}  |  Params: {y.shape}')


In [ ]:
# Quick training (100 epochs for demo)
import subprocess
result = subprocess.run(
    ['python', '../src/emulator/train.py', '--config', '../configs/emulator_config.yaml'],
    capture_output=True, text=True
)
print(result.stdout[-1000:])


In [ ]:
# Evaluate: compare emulated vs true spectrum for a held-out sample
model, normalizer = load_emulator('../results/models/emulator_best.pt')

idx = 42  # sample index
true_params = y[idx]  # [spin, r_bp, beta, phi, inclination]
true_spectrum = X[idx, :, 1:]  # (N_energy, 2): pol_frac, pol_angle
pred_spectrum = model.predict_spectrum(true_params, normalizer)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = ['Polarization Fraction [%]', 'Polarization Angle [°]']
for i, ax in enumerate(axes):
    ax.plot(np.log10(ENERGY_BINS), true_spectrum[:, i], 'k-', label='True')
    ax.plot(np.log10(ENERGY_BINS), pred_spectrum[:, i], 'r--', label='Emulator')
    ax.set_xlabel('log(Energy/keV)')
    ax.set_ylabel(labels[i])
    ax.legend()
    ax.grid(alpha=0.3)
plt.suptitle(f'spin={true_params[0]:.2f}, r_bp={true_params[1]:.1f}, β={true_params[2]:.1f}°')
plt.tight_layout()
plt.savefig('../results/figures/emulator_comparison.png', dpi=150)
plt.show()
